# YOLO-style detector

Redmon, Divvala, Girshick, Farhadi, *You Only Look Once: Unified, Real-Time Object Detection*, CVPR 2016 ([arXiv:1506.02640](https://arxiv.org/abs/1506.02640)).

A simplified single-scale, single-anchor, single-class version: divide the image into a 7x7 grid, have each cell directly regress objectness + box geometry in one forward pass. See `model.py` for the target-assignment/loss details (verified against hand-computed examples) and `README.md` for the simplifications relative to the full paper.

This notebook trains a `YOLOModel` to detect pedestrians in real Penn-Fudan photos.

In [ ]:
import sys
sys.path.insert(0, '../..')
sys.path.insert(0, '.')

import torch
import matplotlib.pyplot as plt
import matplotlib.patches as patches

from cnn_playground.data import load_penn_fudan_detection, penn_fudan_collate
from cnn_playground.device import resolve_device
from cnn_playground.utils.seed import set_seed
from model import GRID_SIZE, YOLOModel, build_targets, decode_box, yolo_loss

set_seed(0)
# device options: 'auto' (default, picks cuda/mps if available), 'cpu', 'cuda', 'mps'
device = resolve_device('auto')
print('device:', device)

In [ ]:
dataset = load_penn_fudan_detection()
n_train = int(len(dataset) * 0.8)
train_ds = torch.utils.data.Subset(dataset, range(n_train))
test_ds = torch.utils.data.Subset(dataset, range(n_train, len(dataset)))
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=8, shuffle=True, collate_fn=penn_fudan_collate)
test_loader = torch.utils.data.DataLoader(test_ds, batch_size=8, shuffle=False, collate_fn=penn_fudan_collate)
print(len(train_ds), 'train images,', len(test_ds), 'test images')
img, boxes = dataset[0]
print('image', img.shape, 'boxes', boxes.shape)

In [ ]:
model = YOLOModel().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

history = {'train_loss': []}
epochs = 40
for epoch in range(epochs):
    model.train()
    last_loss = None
    for imgs, boxes_list in train_loader:
        imgs = imgs.to(device)
        targets = torch.stack([build_targets(b) for b in boxes_list]).to(device)
        opt.zero_grad()
        pred = model(imgs)
        loss = yolo_loss(pred, targets)
        loss.backward()
        opt.step()
        last_loss = loss.item()
    history['train_loss'].append(last_loss)

print(f"final train loss: {history['train_loss'][-1]:.3f}")

In [ ]:
plt.plot(history['train_loss']); plt.title('train loss'); plt.xlabel('epoch'); plt.show()

In [ ]:
model.eval()
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
with torch.no_grad():
    for i in range(3):
        img, real_boxes = test_ds[i]
        pred = model(img.unsqueeze(0).to(device))[0].cpu()
        flat = pred[0].flatten()
        best = flat.argmax().item()
        row, col = best // GRID_SIZE, best % GRID_SIZE
        pred_box = decode_box(pred[:, row, col], row, col)

        ax = axes[i]
        ax.imshow(img.permute(1, 2, 0))
        for rb in real_boxes:
            x0, y0, x1, y1 = rb.tolist()
            ax.add_patch(patches.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, edgecolor='lime', linewidth=2, label='real'))
        x0, y0, x1, y1 = pred_box
        ax.add_patch(patches.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, edgecolor='red', linewidth=2, label='predicted'))
        ax.set_title(f'image {i}'); ax.axis('off')
fig.tight_layout()
plt.show()